<a href="https://colab.research.google.com/github/billurbektas/TP_JSDM/blob/main/Calanda_JSDM/R/02_model/06_sjsdm_experiments_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# sjSDM Model Experiments Pipeline

**Experiments:**
1. Full grid search (spatial_form x alpha x lambda)
2. Decoupled lambda sensitivity with fixed spatial form
3. ANOVA sampling saturation
4. Model fit sampling saturation
5. Drop-one environmental variable
6. k-fold CV (species AUC, train-test gap, site log-loss)

Each run extracts: overall R², full model E/S/C, species-level R²/E/S/C, site-level R²/E/S/C

*The analyses are conducted with Google Colab GPU model A100 with default RAM settings (no high RAM enabled). This is important for reproducibility because at low sampling values different models can give different results.*

## Setup: Install packages & mount Drive

The Google drive mounting needs to be conducted when the runtime is configured for Python 3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

From here on, the runtime needs to be changed to R.

In [ ]:
install.packages("sjSDM")
install.packages("tidyverse")
install.packages("conflicted")
install.packages("pROC")

In [ ]:
library(sjSDM)
install_sjSDM(version = "gpu")


In [ ]:
# Verify GPU
system("nvidia-smi", intern = TRUE)
system("/root/.local/share/r-miniconda/bin/conda run -n r-sjsdm pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --force-reinstall")

Here, the runtime needs to be restarted. On the above menu: **Run all --> Restart session**

In [ ]:
library(reticulate)
use_condaenv("r-sjsdm", required = TRUE)

torch <- import("torch")
cat("PyTorch version:", torch$`__version__`, "\n")
cat("CUDA available:", torch$cuda$is_available(), "\n")
cat("CUDA version:", torch$version$cuda, "\n")

## Load libraries & data

In [ ]:
library(sjSDM)
library(tidyverse)
library(conflicted)
library(pROC)

conflict_prefer("select", "dplyr")
conflict_prefer("filter", "dplyr")
conflict_prefer("intersect", "base")

set.seed(42)
seed = 42
session_info = sessionInfo()

In [3]:
 # ── Fix trailing space bug in sjSDM:::get_null_ll ──
  # Line 346 of anova.R has: inherits(object, "spatial ") — trailing space
  # means spatial models never get spatial_formula = ~0 in the null model.
  # This patch is a 1-character fix applied at runtime.

  local({
    # Get the original function and its environment (sjSDM namespace)
    original = sjSDM:::get_null_ll
    ns_env   = environment(original)

    # Build patched version — identical except "spatial " → "spatial"
    patched = function(object, verbose = TRUE, ...) {
      object_tmp = object
      object_tmp$settings$se = FALSE

      if (inherits(object, "spatial")) {
        null_model = update(object_tmp, env_formula = ~1, spatial_formula = ~0,
                            biotic = bioticStruct(diag = TRUE), verbose = verbose)
        null_pred  = predict(null_model)
      } else {
        null_model = update(object_tmp, env_formula = ~1,
                            biotic = bioticStruct(diag = TRUE), verbose = verbose)
        null_pred  = predict(null_model)
      }

      if (object$family$family$family == "binomial") {
        null_m = stats::dbinom(object$data$Y, 1, null_pred, log = TRUE)
      } else if (object$family$family$family == "poisson") {
        null_m = stats::dpois(object$data$Y, null_pred, log = TRUE)
      } else if (object$family$family$family == "nbinom") {
        check_module()
        torch = pkg.env$torch
        theta = object$theta
        theta = 1.0 / (softplus(theta) + 0.0001)
        theta = matrix(theta, nrow = nrow(null_pred), ncol = ncol(null_pred), byrow = TRUE)
        probs = (1.0 - theta / (theta + null_pred)) + 0.0001
        probs = ifelse(probs < 0.0, 0.0, probs)
        probs = ifelse(probs > 1.0, 1.0 - 0.0001, probs)
        theta = torch$tensor(theta, dtype = torch$float32)
        probs = torch$tensor(probs, dtype = torch$float32)
        YT    = torch$tensor(object$data$Y, dtype = torch$float32)
        null_m = force_r(torch$distributions$NegativeBinomial(
          total_count = theta, probs = probs)$log_prob(YT)$cpu()$data$numpy())
      } else if (object$family$family$family == "gaussian") {
        null_m = sapply(1:ncol(object$data$Y), function(i)
          stats::dnorm(object$data$Y[,i], null_pred[,i],
                       sd = exp(null_model$theta)[i], log = TRUE))
      }
      return(null_m)
    }

    # Set environment to sjSDM namespace so it can see pkg.env, check_module, etc.
    environment(patched) = ns_env

    # Replace in namespace
    assignInNamespace("get_null_ll", patched, ns = "sjSDM")
  })


In [4]:
# ============================================================================
# USER SETTINGS
# ============================================================================

# Google Drive paths
drive_base = "/content/drive/MyDrive/A_ETH/Calanda_JSDM"
data_file = file.path(drive_base, "data_calanda_jsdm_2026-03-06.rds")
results_dir = file.path(drive_base, "results")

# Model settings
device = "gpu"
iterations = 700L
learning_rate = 0.01
sampling_fit = 1000L
anova_samples_fast = 1000L

# Environmental formula
env_formula = as.formula("~summer_temp + fdd + et.annual + slope + rocks_cover +
  trees_cover + shrubs_cover + soil_depth_var +
  tpi + flowdir + nutrient + disturbance")
print(env_formula)

# Toggle experiments
run_exp1 = TRUE   # grid search, to determine alpha and global lambda values
run_exp2 = TRUE   # decoupled lambdas, to determine specific lambda values
run_exp3 = TRUE   # ANOVA saturation, to see species-level stability of variance partioning
run_exp4 = TRUE   # FIT saturation, showed that increasing model fit sampling does not improve
run_exp5 = TRUE   # Drop-one environmental variable
run_exp6 = TRUE   # k-fold CV

# Representative config for Exp 3-5
rep_spatial = "LIN_XY_XY" # updated after the Exp 1-2, space do not gain much with DNN but it might compete for the environment with more complex model.
rep_alpha = 1 # updated after the Exp 1-2, space is too much penalized with alpha = 0
rep_lambda = 0.01 # updated after the Exp 1-2, stable zone for full model R2s.
rep_lambda_env = 0.001 # updated after the Exp 2 # To assure that environment effects remain strong, if not biotic captures it.

~summer_temp + fdd + et.annual + slope + rocks_cover + trees_cover + 
    shrubs_cover + soil_depth_var + tpi + flowdir + nutrient + 
    disturbance


In [ ]:
# Load data
cat("Loading data from:", data_file, "\n")
data_calanda_jsdm = readRDS(data_file)
X = data_calanda_jsdm$X
Y = data_calanda_jsdm$Y

cat("X:", nrow(X), "sites x", ncol(X), "predictors\n")
cat("Y:", nrow(Y), "sites x", ncol(Y), "species\n")
cat("X columns:", paste(colnames(X), collapse = ", "), "\n")

altitude = X[, "altitude"]
XY = X[, c("Latitude", "Longitude")]
richness = rowSums(Y)
env_cols = all.vars(env_formula)
Xenv = X[, env_cols, drop = FALSE]

# Create results directories on Drive
for (d in c("results", "results/runs", "results/decoupled",
            "results/dropone", "results/cv", "results/fit_saturation",
            "results/anova_saturation", "results/anova_repeatability")) {
  dir.create(file.path(drive_base, d), showWarnings = FALSE, recursive = TRUE)
}

## Core functions

In [6]:
fit_sjsdm = function(Y_train, Xenv_train, XY_train,
                       env_form = env_formula,
                       spatial_form = "LIN_XY_XY",
                       lambda_env = 0.001, alpha_env = 1,
                       lambda_sp = 0.01, alpha_sp = 1,
                       lambda_bio = 0.01, alpha_bio = 1) {

    env_component = do.call(linear, list(
      data = Xenv_train,
      formula = env_form,
      lambda = lambda_env,
      alpha = alpha_env
    ))

    if (spatial_form == "DNN") {
      sp_component = DNN(
        XY_train,
        formula = ~0 + .,
        hidden = c(30L, 30L),
        activation = "relu",
        bias = FALSE,
        lambda = lambda_sp, alpha = alpha_sp
      )
    } else if (spatial_form == "LIN_XY_XY") {
      sp_component = do.call(linear, list(
        data = XY_train,
        formula = ~0 + Latitude + Longitude + I(Latitude * Longitude),
        lambda = lambda_sp,
        alpha = alpha_sp
      ))
    } else {
      stop("Unknown spatial_form: ", spatial_form)
    }

    sjSDM(
      Y = Y_train,
      env = env_component,
      spatial = sp_component,
      biotic = bioticStruct(lambda = lambda_bio, alpha = alpha_bio,
                            df = ncol(Y_train), reg_on_Cov = FALSE),
      iter = iterations,
      device = device,
      learning_rate = learning_rate,
      sampling = sampling_fit,
      control = sjSDMControl(
        RMSprop(weight_decay = 0.0),
        scheduler = 5L,
        early_stopping_training = 25L,
        lr_reduce_factor = 0.9
      ),
      se = FALSE
    )
  }

In [7]:
extract_partition = function(model, anova_samples = anova_samples_fast) {

    R2 = Rsquared(model, verbose = FALSE)
    an = anova(model, verbose = FALSE, samples = anova_samples)
    res = internalStructure(an, fractions = "proportional",
                            Rsquared = "McFadden", plot = FALSE, negatives = "floor")

    # --- Species-level (r2 already in output) ---
    sp = res$internals$Species

    # --- Site-level ---
    si = res$internals$Sites

    # --- Summary ---
    summary_row = tibble(
      # Model-level
      overall_R2 = R2,
      # Species
      species_R2_mean = mean(sp$r2, na.rm = TRUE),
      species_E_mean = mean(sp$env, na.rm = TRUE),
      species_S_mean = mean(sp$spa, na.rm = TRUE),
      species_C_mean = mean(sp$codist, na.rm = TRUE),
      species_R2_median = median(sp$r2, na.rm = TRUE),
      species_E_median = median(sp$env, na.rm = TRUE),
      species_S_median = median(sp$spa, na.rm = TRUE),
      species_C_median = median(sp$codist, na.rm = TRUE),
      # Sites
      site_R2_mean = mean(si$r2, na.rm = TRUE),
      site_E_mean = mean(si$env, na.rm = TRUE),
      site_S_mean = mean(si$spa, na.rm = TRUE),
      site_C_mean = mean(si$codist, na.rm = TRUE),
      site_R2_median = median(si$r2, na.rm = TRUE),
      site_E_median = median(si$env, na.rm = TRUE),
      site_S_median = median(si$spa, na.rm = TRUE),
      site_C_median = median(si$codist, na.rm = TRUE)
    )

    list(
      R2 = R2,
      anova = an,
      internal_structure = res,
      species = sp,
      sites = si,
      summary = summary_row
    )
  }


In [8]:
build_stratified_folds = function(k = 10, alt, tree_cover, shrub_cover,
                                    disturbance, nutrient, richness,
                                    site_names = NULL,
                                    woody_thresh = 0, # because the values are z-score standardized this means that we are taking above average values.
                                    seednum = seed,
                                    plot = TRUE) {
    n = length(alt)

    set.seed(seednum)

    elev_bin = as.integer(cut(alt,
      breaks = quantile(alt, probs = seq(0, 1, length.out = 6)),
      include.lowest = TRUE))

    woody_class = ifelse((tree_cover + shrub_cover) > woody_thresh, 1L, 0L)

    landuse_score = scale(disturbance)[, 1] + scale(nutrient)[, 1]
    landuse_bin = as.integer(cut(landuse_score,
      breaks = quantile(landuse_score, probs = seq(0, 1, length.out = 4), na.rm = TRUE),
      include.lowest = TRUE))
    landuse_bin[is.na(landuse_bin)] = 1L

    richness_bin = as.integer(cut(richness,
      breaks = quantile(richness, probs = seq(0, 1, length.out = 4)),
      include.lowest = TRUE))

    strata = paste(elev_bin, woody_class, landuse_bin, richness_bin, sep = "_")
    strata_factor = as.factor(strata)
    fold_ids = rep(NA_integer_, n) #- Creates an empty vector of fold IDs for all n sites
    if (!is.null(site_names)) names(fold_ids) = site_names


  # - For each stratum:
  #  - idx = which sites belong to this stratum (e.g. 7 sites)
  #  - perm = sample(seq_len(k)) = random permutation of 1-10, e.g. [6, 3, 9, 1, 7, 4, 10, 2, 8, 5]
  #  - rep(perm, length.out = 7) = [6, 3, 9, 1, 7, 4, 10] — assigns the 7 sites to folds in that random order, recycling if the stratum had more than 10 sites

    for (s in levels(strata_factor)) {
      idx = which(strata_factor == s)
      perm = sample(seq_len(k))
      fold_ids[idx] = rep(perm, length.out = length(idx))
    }

    if (plot) {
      cat("\n--- Counts per elevation bin per fold ---\n")
      print(table(fold = fold_ids, elev_bin = elev_bin))

      cat("\n--- Counts per land-use bin per fold ---\n")
      print(table(fold = fold_ids, landuse_bin = landuse_bin))

      cat("\n--- Counts per richness bin per fold ---\n")
      print(table(fold = fold_ids, richness_bin = richness_bin))

      cat("\n--- Counts per woody class per fold ---\n")
      print(table(fold = fold_ids, woody_class = woody_class))

      cat("\n--- Min/max altitude and woody fraction per fold ---\n")
      fold_diag = tibble(fold = fold_ids, altitude = alt, woody = woody_class) %>%
        group_by(fold) %>%
        summarise(
          n = n(),
          alt_min = min(altitude),
          alt_max = max(altitude),
          woody_frac = mean(woody),
          .groups = "drop"
        )
      print(as.data.frame(fold_diag))

      # --- PCA plot ---
      df_strata = tibble(
        altitude = scale(alt)[, 1],
        woody_cover = scale(tree_cover + shrub_cover)[, 1],
        landuse_score = scale(landuse_score)[, 1],
        richness = scale(richness)[, 1],
        fold = factor(fold_ids),
        stratum = strata_factor
      )

      pca_strata = prcomp(df_strata %>% select(altitude, woody_cover, landuse_score, richness))
      df_pca = tibble(
        PC1 = pca_strata$x[, 1],
        PC2 = pca_strata$x[, 2],
        fold = df_strata$fold
      )

      var_exp = round(100 * summary(pca_strata)$importance[2, 1:2], 1)

      loadings = as.data.frame(pca_strata$rotation[, 1:2])
      loadings$variable = rownames(loadings)
      arrow_scale = max(abs(df_pca$PC1), abs(df_pca$PC2)) * 0.8

      p = ggplot(df_pca, aes(x = PC1, y = PC2, color = fold)) +
        geom_point(alpha = 0.6, size = 2) +
        geom_segment(data = loadings,
                     aes(x = 0, y = 0, xend = PC1 * arrow_scale, yend = PC2 * arrow_scale),
                     inherit.aes = FALSE,
                     arrow = arrow(length = unit(0.2, "cm")),
                     color = "grey30", linewidth = 0.6) +
        geom_text(data = loadings,
                  aes(x = PC1 * arrow_scale * 1.12, y = PC2 * arrow_scale * 1.12, label = variable),
                  inherit.aes = FALSE, size = 3.2, color = "grey20") +
        labs(title = "Stratified folds in strata-variable PCA space",
             x = paste0("PC1 (", var_exp[1], "%)"),
             y = paste0("PC2 (", var_exp[2], "%)"),
             color = "Fold") +
        theme_bw() +
        theme(legend.position = "right")

      print(p)
    }

    folds = lapply(seq_len(k), function(i) {
      idx = which(fold_ids == i)
      if (!is.null(site_names)) names(idx) = site_names[idx]
      idx
    })
    folds
  }


In [ ]:
# Check the balance of the design
  k = 10
  cat("Config:", rep_spatial, "alpha=", rep_alpha, "lambda=", rep_lambda, "\n")
  flush.console()

  folds_file = file.path(drive_base, "results", "cv", "stratified_folds.rds")

  if (file.exists(folds_file)) {
    cat("Cached: stratified folds\n")
    flush.console()
    folds = readRDS(folds_file)
  } else {
    folds = build_stratified_folds(
      k = k, alt = altitude,
      tree_cover = Xenv[, "trees_cover"],
      shrub_cover = Xenv[, "shrubs_cover"],
      disturbance = Xenv[, "disturbance"],
      nutrient = Xenv[, "nutrient"],
      richness = richness,
      site_names = rownames(Y)
    )
    saveRDS(folds, folds_file)
    cat("Saved:", basename(folds_file), "\n")
    flush.console()
  }

  str(folds)
  cat("Fold sizes:", paste(sapply(folds, length), collapse = ", "), "\n\n")
  flush.console()

## Experiment 1: Full grid search

In [ ]:
if (run_exp1) {
  cat("\n", strrep("=", 60), "\n")
    cat("EXPERIMENT 1: Full grid search\n")
    cat(strrep("=", 60), "\n")

    grid = expand.grid(
      spatial_form = c("DNN", "LIN_XY_XY"),
      alpha_common = c(0, 0.5, 1),
      lambda_common = c(0.001, 0.01, 0.1, 0.5, 1),
      stringsAsFactors = FALSE
    ) %>%
      mutate(run_id = paste0(spatial_form, "_a", alpha_common, "_l", lambda_common))

    cat("Total runs:", nrow(grid), "\n\n")

    print(grid) # check the grid design
  }

In [ ]:
if (run_exp1) {
  exp1_summaries = list()

    for (i in seq_len(nrow(grid))) {

      cfg = grid[i, ]
      run_file = file.path(drive_base, "results", "runs", paste0(cfg$run_id, ".rds"))

      if (file.exists(run_file)) {
        cat("[", i, "/", nrow(grid), "] Cached:", cfg$run_id, "\n")
        flush.console()
        run_data = readRDS(run_file)
      } else {
        cat("[", i, "/", nrow(grid), "] Fitting:", cfg$run_id, "\n")
        flush.console()

        t0 = Sys.time()
        model = fit_sjsdm(
          Y_train = Y, Xenv_train = Xenv, XY_train = XY,
          spatial_form = cfg$spatial_form,
          env_form = env_formula,
          lambda_env = cfg$lambda_common, alpha_env = cfg$alpha_common,
          lambda_sp = cfg$lambda_common, alpha_sp = cfg$alpha_common,
          lambda_bio = cfg$lambda_common, alpha_bio = cfg$alpha_common
        )
        t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  Fit done in", t_fit, "min\n")
        flush.console()

        cat("[", i, "/", nrow(grid), "] Variance partitioning:", cfg$run_id, "\n")
        flush.console()
        t0 = Sys.time()
        partition = extract_partition(model)
        t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  VP done in", t_vp, "min\n")
        flush.console()

        run_data = list(config = cfg, model = model, partition = partition)
        saveRDS(run_data, run_file)
        cat("  -> Saved (total:", as.numeric(t_fit) + as.numeric(t_vp), "min)\n")
        flush.console()
      }

      exp1_summaries[[i]] = bind_cols(cfg, run_data$partition$summary)
    }

    summary_exp1 = bind_rows(exp1_summaries)
    write_csv(summary_exp1, file.path(drive_base, "results", "summary_exp1_grid.csv"))
    cat("\nSaved summary_exp1_grid.csv\n")
    flush.console()
    print(summary_exp1)
  }

## Experiment 2: Decoupled lambda sensitivity

In [ ]:
cat("\n", strrep("=", 60), "\n")
    cat("EXPERIMENT 2: Decoupled lambda sensitivity\n")
    cat(strrep("=", 60), "\n")

    lambda_anchor = 0.01
    lambda_values = c(0.001, 0.01, 0.1)
    alpha_values  = c(0, 0.5, 1)

    grid_decoupled = expand.grid(
      spatial_form = "LIN_XY_XY",
      vary         = c("lambda_env", "lambda_sp", "lambda_bio"),
      lambda_vary  = lambda_values,
      alpha_common = alpha_values,
      stringsAsFactors = FALSE
    ) %>%
      mutate(
        lambda_env = ifelse(vary == "lambda_env", lambda_vary, lambda_anchor),
        lambda_sp  = ifelse(vary == "lambda_sp",  lambda_vary, lambda_anchor),
        lambda_bio = ifelse(vary == "lambda_bio", lambda_vary, lambda_anchor),
        run_id = paste0("decoupled_", spatial_form, "_", vary, "_", lambda_vary, "_a", alpha_common)
      ) %>%
      filter(lambda_vary != lambda_anchor)

    cat("Total runs:", nrow(grid_decoupled), "\n\n")
    print(grid_decoupled)

In [ ]:
if (run_exp2) {

    exp2_summaries = list()

    for (i in seq_len(nrow(grid_decoupled))) {

      cfg = grid_decoupled[i, ]
      run_file = file.path(drive_base, "results", "decoupled", paste0(cfg$run_id, ".rds"))

      if (file.exists(run_file)) {
        cat("[", i, "/", nrow(grid_decoupled), "] Cached:", cfg$run_id, "\n")
        flush.console()
        run_data = readRDS(run_file)
      } else {
        cat("[", i, "/", nrow(grid_decoupled), "] Fitting:", cfg$run_id, "\n")
        flush.console()

        model = fit_sjsdm(
          Y_train = Y, Xenv_train = Xenv, XY_train = XY,
          spatial_form = cfg$spatial_form,
          lambda_env = cfg$lambda_env, alpha_env = cfg$alpha_common,
          lambda_sp  = cfg$lambda_sp,  alpha_sp  = cfg$alpha_common,
          lambda_bio = cfg$lambda_bio, alpha_bio = cfg$alpha_common
        )

        partition = extract_partition(model)

        run_data = list(config = cfg, model = model, partition = partition)
        saveRDS(run_data, run_file)
        cat("  -> Saved\n")
        flush.console()
      }

      exp2_summaries[[i]] = bind_cols(
        cfg %>% select(run_id, spatial_form, vary, lambda_vary, alpha_common,
                       lambda_env, lambda_sp, lambda_bio),
        run_data$partition$summary
      )
    }

    summary_exp2 = bind_rows(exp2_summaries)
    write_csv(summary_exp2, file.path(drive_base, "results", "summary_exp2_decoupled.csv"))
    cat("\nSaved summary_exp2_decoupled.csv\n")
    flush.console()
    print(summary_exp2)
  }


## Experiment 2b: Reproducibility check for two identical runs with same configuration



In [ ]:
 cat("\n", strrep("=", 60), "\n")
  cat("REPRODUCIBILITY CHECK: Two identical LIN_XY_XY a=1 l=0.01 fits\n")
  cat(strrep("=", 60), "\n")

  repro_config = list(spatial_form = "LIN_XY_XY", alpha = 1, lambda = 0.01)

  for (run_i in 1:2) {
    run_file = file.path(drive_base, "results", paste0("repro_check_run", run_i, ".rds"))

    if (file.exists(run_file)) {
      cat("[Run", run_i, "] Cached\n")
      flush.console()
    } else {
      cat("[Run", run_i, "] Fitting...\n")
      flush.console()

      t0 = Sys.time()
      model = fit_sjsdm(
        Y_train = Y, Xenv_train = Xenv, XY_train = XY,
        spatial_form = repro_config$spatial_form,
        env_form = env_formula,
        lambda_env = repro_config$lambda, alpha_env = repro_config$alpha,
        lambda_sp  = repro_config$lambda, alpha_sp  = repro_config$alpha,
        lambda_bio = repro_config$lambda, alpha_bio = repro_config$alpha
      )
      t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
      cat("  Fit done in", t_fit, "min\n")
      flush.console()

      cat("[Run", run_i, "] Variance partitioning...\n")
      flush.console()
      t0 = Sys.time()
      partition = extract_partition(model)
      t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
      cat("  VP done in", t_vp, "min\n")
      flush.console()

      run_data = list(config = repro_config, model = model, partition = partition)
      saveRDS(run_data, run_file)
      cat("  -> Saved (total:", as.numeric(t_fit) + as.numeric(t_vp), "min)\n")
      flush.console()
    }
  }

  # --- Compare the two runs ---
  cat("\n", strrep("-", 60), "\n")
  cat("COMPARISON\n")
  cat(strrep("-", 60), "\n")

  r1 = readRDS(file.path(drive_base, "results", "repro_check_run1.rds"))
  r2 = readRDS(file.path(drive_base, "results", "repro_check_run2.rds"))

  # Overall R2
  cat(sprintf("\nOverall R2:  Run1=%.6f  Run2=%.6f  diff=%.6f\n",
              r1$partition$R2, r2$partition$R2,
              r2$partition$R2 - r1$partition$R2))

  # Species-level correlations
  cat("\nSpecies-level correlations (Run1 vs Run2):\n")
  for (col in c("env", "spa", "codist", "r2")) {
    r = cor(r1$partition$species[, col], r2$partition$species[, col], use = "complete.obs")
    cat(sprintf("  %-7s r = %.4f\n", col, r))
  }

  # Site-level correlations
  cat("\nSite-level correlations (Run1 vs Run2):\n")
  for (col in c("env", "spa", "codist", "r2")) {
    r = cor(r1$partition$sites[, col], r2$partition$sites[, col], use = "complete.obs")
    cat(sprintf("  %-7s r = %.4f\n", col, r))
  }

  cat("\n(If reproducible: diffs ~ 0, correlations ~ 1)\n")

## Experiment 3: Fix model fitting sample size and test anova sampling sizes

In [ ]:
if (run_exp3) {

        cat("\n", strrep("=", 60), "\n")
        cat("EXPERIMENT 6: ANOVA sampling saturation (fixed model fit)\n")
        cat(strrep("=", 60), "\n")

        fit_sampling = 5000L
        anova_sizes = c(5000L, 10000L, 20000L, 30000L, 50000L, 80000L, 100000L, 130000L, 150000L, 180000L)
        ref_anova_size = max(anova_sizes)

        # Build a tag for filenames
        param_tag = paste0("a", rep_alpha, "_l", rep_lambda, "_le", rep_lambda_env)

        cat("Model fitting sampling:", fit_sampling, "\n")
        cat("Alpha:", rep_alpha, "| Lambda:", rep_lambda, "| Lambda_env:", rep_lambda_env, "\n")
        cat("ANOVA sample sizes:", paste(anova_sizes, collapse = ", "), "\n")
        cat("Reference:", ref_anova_size, "\n\n")
        flush.console()

        # --- Step 1: Fit a single model (or load cached) ---
        model_file = file.path(drive_base, "results", "anova_saturation",
                               paste0("exp6_model_fit", fit_sampling, "_", param_tag, ".rds"))

        if (file.exists(model_file)) {
          cat("Cached: model (sampling =", fit_sampling, ",", param_tag, ")\n")
          flush.console()
          model_exp6 = readRDS(model_file)
        } else {
          cat("Fitting model (sampling =", fit_sampling, ",", param_tag, ")...\n")
          flush.console()

          t0 = Sys.time()
          model_exp6 = sjSDM(
            Y = Y,
            env = do.call(linear, list(
              data = Xenv,
              formula = env_formula,
              lambda = rep_lambda_env, alpha = rep_alpha
            )),
            spatial = do.call(linear, list(
              data = XY,
              formula = ~0 + Latitude + Longitude + I(Latitude * Longitude),
              lambda = rep_lambda, alpha = rep_alpha
            )),
            biotic = bioticStruct(lambda = rep_lambda, alpha = rep_alpha,
                                  df = ncol(Y), reg_on_Cov = FALSE),
            iter = iterations,
            device = device,
            learning_rate = learning_rate,
            sampling = fit_sampling,
            control = sjSDMControl(
              RMSprop(weight_decay = 0.0),
              scheduler = 5L,
              early_stopping_training = 25L,
              lr_reduce_factor = 0.9
            ),
            se = FALSE
          )
          t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
          cat("  Fit done in", t_fit, "min\n")
          flush.console()

          # Attach hyperparameters as attributes
          attr(model_exp6, "hyperparameters") = list(
            alpha = rep_alpha,
            lambda = rep_lambda,
            lambda_env = rep_lambda_env,
            fit_sampling = fit_sampling,
            iterations = iterations,
            learning_rate = learning_rate
          )

          saveRDS(model_exp6, model_file)
          cat("  -> Saved model\n")
          flush.console()
        }

        # --- Step 2: Run anova at each sample size ---
        exp6_results = list()

        for (ns in anova_sizes) {
          vp_file = file.path(drive_base, "results", "anova_saturation",
                              paste0("exp6_anova_", ns, "_", param_tag, ".rds"))

          if (file.exists(vp_file)) {
            cat("Cached: anova_samples =", ns, "\n")
            flush.console()
            exp6_results[[as.character(ns)]] = readRDS(vp_file)
          } else {
            cat("Variance partitioning: anova_samples =", ns, "...\n")
            flush.console()

            t0 = Sys.time()
            partition = extract_partition(model_exp6, anova_samples = as.integer(ns))
            t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
            cat("  VP done in", t_vp, "min\n")
            flush.console()

            run_data = list(
              partition = partition,
              anova_samples = ns,
              vp_time_min = as.numeric(t_vp),
              alpha = rep_alpha,
              lambda = rep_lambda,
              lambda_env = rep_lambda_env
            )
            saveRDS(run_data, vp_file)
            cat("  -> Saved (", t_vp, "min)\n")
            flush.console()

            exp6_results[[as.character(ns)]] = run_data
          }
        }

        # --- Step 3: Summary table (correlations + MAD vs reference) ---
        ref = exp6_results[[as.character(ref_anova_size)]]$partition

        exp6_table = tibble(
          anova_samples = anova_sizes,
          alpha = rep_alpha,
          lambda = rep_lambda,
          lambda_env = rep_lambda_env
        )

        for (idx in seq_along(anova_sizes)) {
          ns = anova_sizes[idx]
          p = exp6_results[[as.character(ns)]]$partition
          t_vp = exp6_results[[as.character(ns)]]$vp_time_min

          exp6_table$overall_R2[idx] = p$R2
          exp6_table$vp_time_min[idx] = ifelse(is.null(t_vp), NA_real_, t_vp)

          # Correlations with reference
          exp6_table$cor_E_species[idx] = cor(p$species$env, ref$species$env, use = "complete.obs")
          exp6_table$cor_S_species[idx] = cor(p$species$spa, ref$species$spa, use = "complete.obs")
          exp6_table$cor_C_species[idx] = cor(p$species$codist, ref$species$codist, use = "complete.obs")
          exp6_table$cor_E_site[idx]    = cor(p$sites$env, ref$sites$env, use = "complete.obs")
          exp6_table$cor_S_site[idx]    = cor(p$sites$spa, ref$sites$spa, use = "complete.obs")
          exp6_table$cor_C_site[idx]    = cor(p$sites$codist, ref$sites$codist, use = "complete.obs")

          # Mean absolute differences with reference
          exp6_table$mad_E_species[idx] = mean(abs(p$species$env - ref$species$env), na.rm = TRUE)
          exp6_table$mad_S_species[idx] = mean(abs(p$species$spa - ref$species$spa), na.rm = TRUE)
          exp6_table$mad_C_species[idx] = mean(abs(p$species$codist - ref$species$codist), na.rm = TRUE)
          exp6_table$mad_E_site[idx]    = mean(abs(p$sites$env - ref$sites$env), na.rm = TRUE)
          exp6_table$mad_S_site[idx]    = mean(abs(p$sites$spa - ref$sites$spa), na.rm = TRUE)
          exp6_table$mad_C_site[idx]    = mean(abs(p$sites$codist - ref$sites$codist), na.rm = TRUE)
        }

        summary_file = file.path(drive_base, "results",
                                 paste0("summary_exp6_anova_saturation_", param_tag, ".csv"))
        write_csv(exp6_table, summary_file)
        cat("\nSaved", basename(summary_file), "\n")
        flush.console()
        print(exp6_table)
      }

##Experiment 4: Fix anova sampling size and test model fitting size

In [ ]:
if (run_exp4) {

        cat("\n", strrep("=", 60), "\n")
        cat("EXPERIMENT 7: Model fit sampling saturation (fixed ANOVA)\n")
        cat(strrep("=", 60), "\n")

        fixed_anova = 30000L
        fit_sizes = c(5000L, 10000L, 20000L, 30000L)
        ref_fit_size = max(fit_sizes)

        # Build a tag for filenames
        param_tag = paste0("a", rep_alpha, "_l", rep_lambda, "_le", rep_lambda_env)

        cat("Fixed ANOVA samples:", fixed_anova, "\n")
        cat("Alpha:", rep_alpha, "| Lambda:", rep_lambda, "| Lambda_env:", rep_lambda_env, "\n")
        cat("Model fit sampling sizes:", paste(fit_sizes, collapse = ", "), "\n")
        cat("Reference:", ref_fit_size, "\n\n")
        flush.console()

        # --- Step 1: Fit models at each sampling size (or load cached) ---
        exp7_models  = list()
        exp7_results = list()

        for (fs in fit_sizes) {
          fs_chr = as.character(fs)

          model_file = file.path(drive_base, "results", "fit_saturation",
                                 paste0("exp7_model_fit", fs, "_", param_tag, ".rds"))

          if (file.exists(model_file)) {
            cat("Cached: model (sampling =", fs, ",", param_tag, ")\n")
            flush.console()
            exp7_models[[fs_chr]] = readRDS(model_file)
          } else {
            cat("Fitting model (sampling =", fs, ",", param_tag, ")...\n")
            flush.console()

            t0 = Sys.time()
            mod = sjSDM(
              Y = Y,
              env = do.call(linear, list(
                data = Xenv,
                formula = env_formula,
                lambda = rep_lambda_env, alpha = rep_alpha
              )),
              spatial = do.call(linear, list(
                data = XY,
                formula = ~0 + Latitude + Longitude + I(Latitude * Longitude),
                lambda = rep_lambda, alpha = rep_alpha
              )),
              biotic = bioticStruct(lambda = rep_lambda, alpha = rep_alpha,
                                    df = ncol(Y), reg_on_Cov = FALSE),
              iter = iterations,
              device = device,
              learning_rate = learning_rate,
              sampling = fs,
              control = sjSDMControl(
                RMSprop(weight_decay = 0.0),
                scheduler = 5L,
                early_stopping_training = 25L,
                lr_reduce_factor = 0.9
              ),
              se = FALSE
            )
            t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
            cat("  Fit done in", t_fit, "min\n")
            flush.console()

            attr(mod, "hyperparameters") = list(
              alpha = rep_alpha,
              lambda = rep_lambda,
              lambda_env = rep_lambda_env,
              fit_sampling = fs,
              iterations = iterations,
              learning_rate = learning_rate
            )

            saveRDS(mod, model_file)
            cat("  -> Saved model\n")
            flush.console()

            exp7_models[[fs_chr]] = mod
          }
        }

        # --- Step 2: Run anova at fixed sample size for each model ---
        for (fs in fit_sizes) {
          fs_chr = as.character(fs)

          vp_file = file.path(drive_base, "results", "fit_saturation",
                              paste0("exp7_anova_fit", fs, "_av", fixed_anova, "_", param_tag, ".rds"))

          if (file.exists(vp_file)) {
            cat("Cached: VP for fit_sampling =", fs, "\n")
            flush.console()
            exp7_results[[fs_chr]] = readRDS(vp_file)
          } else {
            cat("Variance partitioning: fit_sampling =", fs, ", anova_samples =", fixed_anova, "...\n")
            flush.console()

            t0 = Sys.time()
            partition = extract_partition(exp7_models[[fs_chr]], anova_samples = fixed_anova)
            t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
            cat("  VP done in", t_vp, "min\n")
            flush.console()

            run_data = list(
              partition = partition,
              fit_sampling = fs,
              anova_samples = fixed_anova,
              vp_time_min = as.numeric(t_vp),
              alpha = rep_alpha,
              lambda = rep_lambda,
              lambda_env = rep_lambda_env
            )
            saveRDS(run_data, vp_file)
            cat("  -> Saved (", t_vp, "min)\n")
            flush.console()

            exp7_results[[fs_chr]] = run_data
          }
        }

        # --- Step 3: Summary table (correlations + MAD vs reference) ---
        ref = exp7_results[[as.character(ref_fit_size)]]$partition

        exp7_table = tibble(
          fit_sampling = fit_sizes,
          anova_samples = fixed_anova,
          alpha = rep_alpha,
          lambda = rep_lambda,
          lambda_env = rep_lambda_env
        )

        for (idx in seq_along(fit_sizes)) {
          fs = fit_sizes[idx]
          p = exp7_results[[as.character(fs)]]$partition
          t_vp = exp7_results[[as.character(fs)]]$vp_time_min

          exp7_table$overall_R2[idx] = p$R2
          exp7_table$vp_time_min[idx] = ifelse(is.null(t_vp), NA_real_, t_vp)

          # Correlations with reference
          exp7_table$cor_E_species[idx] = cor(p$species$env, ref$species$env, use = "complete.obs")
          exp7_table$cor_S_species[idx] = cor(p$species$spa, ref$species$spa, use = "complete.obs")
          exp7_table$cor_C_species[idx] = cor(p$species$codist, ref$species$codist, use = "complete.obs")
          exp7_table$cor_E_site[idx]    = cor(p$sites$env, ref$sites$env, use = "complete.obs")
          exp7_table$cor_S_site[idx]    = cor(p$sites$spa, ref$sites$spa, use = "complete.obs")
          exp7_table$cor_C_site[idx]    = cor(p$sites$codist, ref$sites$codist, use = "complete.obs")

          # Mean absolute differences with reference
          exp7_table$mad_E_species[idx] = mean(abs(p$species$env - ref$species$env), na.rm = TRUE)
          exp7_table$mad_S_species[idx] = mean(abs(p$species$spa - ref$species$spa), na.rm = TRUE)
          exp7_table$mad_C_species[idx] = mean(abs(p$species$codist - ref$species$codist), na.rm = TRUE)
          exp7_table$mad_E_site[idx]    = mean(abs(p$sites$env - ref$sites$env), na.rm = TRUE)
          exp7_table$mad_S_site[idx]    = mean(abs(p$sites$spa - ref$sites$spa), na.rm = TRUE)
          exp7_table$mad_C_site[idx]    = mean(abs(p$sites$codist - ref$sites$codist), na.rm = TRUE)
        }

        summary_file = file.path(drive_base, "results",
                                 paste0("summary_exp7_fit_saturation_", param_tag, ".csv"))
        write_csv(exp7_table, summary_file)
        cat("\nSaved", basename(summary_file), "\n")
        flush.console()
        print(exp7_table)
      }


## Experiment 5: Drop-one environmental variable

In [ ]:
if (run_exp5) {

      cat("\n", strrep("=", 60), "\n")
      cat("EXPERIMENT 3: Drop-one environmental variable\n")
      cat(strrep("=", 60), "\n")

      exp3_fit_sampling  = 5000L
      exp3_anova_samples = 80000L

      # Build a tag for filenames
      param_tag = paste0("a", rep_alpha, "_l", rep_lambda, "_le", rep_lambda_env)

      cat("Config:", rep_spatial, "\n")
      cat("Alpha:", rep_alpha, "| Lambda:", rep_lambda, "| Lambda_env:", rep_lambda_env, "\n")
      cat("Fit sampling:", exp3_fit_sampling, "| ANOVA samples:", exp3_anova_samples, "\n")
      cat("Param tag:", param_tag, "\n\n")
      flush.console()

      # --- Base model (full, all predictors) ---
      base_file = file.path(drive_base, "results", "dropone",
                            paste0("dropone_BASE_", param_tag, ".rds"))

      if (file.exists(base_file)) {
        cat("[BASE] Cached:", basename(base_file), "\n")
        flush.console()
        base_data = readRDS(base_file)
      } else {
        cat("[BASE] Fitting full model...\n")
        flush.console()

        t0 = Sys.time()
        model_base = sjSDM(
          Y = Y,
          env = do.call(linear, list(
            data = Xenv,
            formula = env_formula,
            lambda = rep_lambda_env, alpha = rep_alpha
          )),
          spatial = do.call(linear, list(
            data = XY,
            formula = ~0 + Latitude + Longitude + I(Latitude * Longitude),
            lambda = rep_lambda, alpha = rep_alpha
          )),
          biotic = bioticStruct(lambda = rep_lambda, alpha = rep_alpha,
                                df = ncol(Y), reg_on_Cov = FALSE),
          iter = iterations,
          device = device,
          learning_rate = learning_rate,
          sampling = exp3_fit_sampling,
          control = sjSDMControl(
            RMSprop(weight_decay = 0.0),
            scheduler = 5L,
            early_stopping_training = 25L,
            lr_reduce_factor = 0.9
          ),
          se = FALSE
        )
        t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  Fit done in", t_fit, "min\n")
        flush.console()

        cat("[BASE] Variance partitioning (samples =", exp3_anova_samples, ")...\n")
        flush.console()
        t0 = Sys.time()
        partition_base = extract_partition(model_base, anova_samples = exp3_anova_samples)
        t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  VP done in", t_vp, "min\n")
        flush.console()

        base_data = list(
          dropped = "NONE",
          partition = partition_base,
          alpha = rep_alpha,
          lambda = rep_lambda,
          lambda_env = rep_lambda_env,
          fit_sampling = exp3_fit_sampling,
          anova_samples = exp3_anova_samples,
          fit_time_min = as.numeric(t_fit),
          vp_time_min = as.numeric(t_vp)
        )
        saveRDS(base_data, base_file)
        cat("  -> Saved (total:", as.numeric(t_fit) + as.numeric(t_vp), "min)\n")
        flush.console()
      }

      # --- Drop-one loop ---
      exp3_summaries = list()

      # Add base model as first row
      exp3_summaries[[1]] = bind_cols(
        tibble(dropped_variable = "NONE"),
        base_data$partition$summary
      )

      for (j in seq_along(env_cols)) {

        pred = env_cols[j]
        run_id = paste0("dropone_", pred)
        run_file = file.path(drive_base, "results", "dropone",
                             paste0(run_id, "_", param_tag, ".rds"))

        if (file.exists(run_file)) {
          cat("[", j, "/", length(env_cols), "] Cached:", pred, "\n")
          flush.console()
          run_data = readRDS(run_file)
        } else {
          cat("[", j, "/", length(env_cols), "] Dropping:", pred, "\n")
          flush.console()

          t0 = Sys.time()
          reduced_cols = setdiff(env_cols, pred)
          reduced_formula = as.formula(paste("~", paste(reduced_cols, collapse = " + ")))

          model = sjSDM(
            Y = Y,
            env = do.call(linear, list(
              data = Xenv[, reduced_cols, drop = FALSE],
              formula = reduced_formula,
              lambda = rep_lambda_env, alpha = rep_alpha
            )),
            spatial = do.call(linear, list(
              data = XY,
              formula = ~0 + Latitude + Longitude + I(Latitude * Longitude),
              lambda = rep_lambda, alpha = rep_alpha
            )),
            biotic = bioticStruct(lambda = rep_lambda, alpha = rep_alpha,
                                  df = ncol(Y), reg_on_Cov = FALSE),
            iter = iterations,
            device = device,
            learning_rate = learning_rate,
            sampling = exp3_fit_sampling,
            control = sjSDMControl(
              RMSprop(weight_decay = 0.0),
              scheduler = 5L,
              early_stopping_training = 25L,
              lr_reduce_factor = 0.9
            ),
            se = FALSE
          )
          t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
          cat("  Fit done in", t_fit, "min\n")
          flush.console()

          cat("[", j, "/", length(env_cols), "] Variance partitioning:", pred,
              "(samples =", exp3_anova_samples, ")\n")
          flush.console()
          t0 = Sys.time()
          partition = extract_partition(model, anova_samples = exp3_anova_samples)
          t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
          cat("  VP done in", t_vp, "min\n")
          flush.console()

          run_data = list(
            dropped = pred,
            partition = partition,
            alpha = rep_alpha,
            lambda = rep_lambda,
            lambda_env = rep_lambda_env,
            fit_sampling = exp3_fit_sampling,
            anova_samples = exp3_anova_samples,
            fit_time_min = as.numeric(t_fit),
            vp_time_min = as.numeric(t_vp)
          )
          saveRDS(run_data, run_file)
          cat("  -> Saved (total:", as.numeric(t_fit) + as.numeric(t_vp), "min)\n")
          flush.console()
        }

        exp3_summaries[[j + 1]] = bind_cols(
          tibble(dropped_variable = pred),
          run_data$partition$summary
        )
      }

      summary_exp3 = bind_rows(exp3_summaries) %>%
        mutate(alpha = rep_alpha, lambda = rep_lambda, lambda_env = rep_lambda_env,
               fit_sampling = exp3_fit_sampling, anova_samples = exp3_anova_samples,
               .before = 1)

      summary_file = paste0("summary_exp3_dropone_", param_tag, ".csv")
      write_csv(summary_exp3, file.path(drive_base, "results", summary_file))
      cat("\nSaved", summary_file, "\n")
      flush.console()
      print(summary_exp3)
    }

## Experiment 6: k-fold CV (species AUC, train-test gap, site log-loss)

In [ ]:
if (run_exp6) {

      cat("\n", strrep("=", 60), "\n")
      cat("EXPERIMENT 4: k-fold cross-validation\n")
      cat(strrep("=", 60), "\n")

      k = 10
      exp4_fit_sampling  = 5000L
      exp4_anova_samples = 80000L

      # Build a tag for filenames
      param_tag = paste0("a", rep_alpha, "_l", rep_lambda, "_le", rep_lambda_env)

      cat("Config:", rep_spatial, "\n")
      cat("Alpha:", rep_alpha, "| Lambda:", rep_lambda, "| Lambda_env:", rep_lambda_env, "\n")
      cat("Fit sampling:", exp4_fit_sampling, "| ANOVA samples:", exp4_anova_samples, "\n")
      cat("Param tag:", param_tag, "\n\n")
      flush.console()

      folds_file = file.path(drive_base, "results", "cv", "stratified_folds.rds")
      if (file.exists(folds_file)) {
        cat("Cached: stratified folds\n")
        flush.console()
        folds = readRDS(folds_file)
      } else {
        folds = build_stratified_folds(
          k = k, alt = altitude,
          tree_cover = Xenv[, "trees_cover"],
          shrub_cover = Xenv[, "shrubs_cover"],
          disturbance = Xenv[, "disturbance"],
          nutrient = Xenv[, "nutrient"],
          richness = richness,
          site_names = rownames(Y)
        )
        saveRDS(folds, folds_file)
        cat("Saved:", basename(folds_file), "\n")
        flush.console()
      }
      cat("Fold sizes:", paste(sapply(folds, length), collapse = ", "), "\n\n")
      flush.console()

      P_oof = matrix(NA, nrow = nrow(Y), ncol = ncol(Y),
                     dimnames = list(rownames(Y), colnames(Y)))
      train_auc_per_fold = matrix(NA, nrow = k, ncol = ncol(Y))

      for (fold_i in seq_len(k)) {

        fold_file = file.path(drive_base, "results", "cv",
                              paste0("fold_", fold_i, "_", param_tag, ".rds"))
        test_idx = folds[[fold_i]]
        train_idx = setdiff(seq_len(nrow(Y)), test_idx)

        if (file.exists(fold_file)) {
          cat("[Fold", fold_i, "/", k, "] Cached:", basename(fold_file), "\n")
          flush.console()
          fold_data = readRDS(fold_file)
        } else {
          cat("[Fold", fold_i, "/", k, "] Fitting...\n")
          flush.console()

          t0 = Sys.time()
          model_fold = sjSDM(
            Y = Y[train_idx, ],
            env = do.call(linear, list(
              data = Xenv[train_idx, , drop = FALSE],
              formula = env_formula,
              lambda = rep_lambda_env, alpha = rep_alpha
            )),
            spatial = do.call(linear, list(
              data = XY[train_idx, , drop = FALSE],
              formula = ~0 + Latitude + Longitude + I(Latitude * Longitude),
              lambda = rep_lambda, alpha = rep_alpha
            )),
            biotic = bioticStruct(lambda = rep_lambda, alpha = rep_alpha,
                                  df = ncol(Y), reg_on_Cov = FALSE),
            iter = iterations,
            device = device,
            learning_rate = learning_rate,
            sampling = exp4_fit_sampling,
            control = sjSDMControl(
              RMSprop(weight_decay = 0.0),
              scheduler = 5L,
              early_stopping_training = 25L,
              lr_reduce_factor = 0.9
            ),
            se = FALSE
          )
          t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
          cat("  Fit done in", t_fit, "min\n")
          flush.console()

          cat("[Fold", fold_i, "/", k, "] Predicting...\n")
          flush.console()

          preds_test = predict(model_fold,
                               newdata = Xenv[test_idx, , drop = FALSE],
                               SP = XY[test_idx, , drop = FALSE])

          preds_train = predict(model_fold,
                                newdata = Xenv[train_idx, , drop = FALSE],
                                SP = XY[train_idx, , drop = FALSE])

          # --- Variance partitioning for this fold ---
          cat("[Fold", fold_i, "/", k, "] Variance partitioning (samples =", exp4_anova_samples, ")...\n")
          flush.console()
          t0_vp = Sys.time()
          partition_fold = extract_partition(model_fold, anova_samples = exp4_anova_samples)
          t_vp = round(difftime(Sys.time(), t0_vp, units = "mins"), 1)
          cat("  VP done in", t_vp, "min\n")
          flush.console()

          fold_data = list(
            fold = fold_i,
            train_idx = train_idx,
            test_idx = test_idx,
            preds_test = preds_test,
            preds_train = preds_train,
            partition = partition_fold,
            alpha = rep_alpha,
            lambda = rep_lambda,
            lambda_env = rep_lambda_env,
            fit_sampling = exp4_fit_sampling,
            anova_samples = exp4_anova_samples,
            fit_time_min = as.numeric(t_fit),
            vp_time_min = as.numeric(t_vp)
          )
          saveRDS(fold_data, fold_file)
          cat("  -> Saved (fit:", t_fit, "min, VP:", t_vp, "min)\n")
          flush.console()
        }

        P_oof[fold_data$test_idx, ] = fold_data$preds_test

        # Train AUC per species for this fold
        for (s in seq_len(ncol(Y))) {
          y_tr = Y[fold_data$train_idx, s]
          p_tr = fold_data$preds_train[, s]
          if (sum(y_tr == 1) >= 2 && sum(y_tr == 0) >= 2) {
            train_auc_per_fold[fold_i, s] = tryCatch(
              as.numeric(auc(roc(y_tr, p_tr, quiet = TRUE))),
              error = function(e) NA
            )
          }
        }
      }

      cat("\nAll folds complete. Computing metrics...\n")
      flush.console()

      # --- Species metrics: test AUC + train-test gap ---
      species_cv = tibble(
        species = colnames(Y),
        n_presences = as.integer(colSums(Y)),
        test_auc = NA_real_,
        train_auc_mean = NA_real_,
        train_test_gap = NA_real_
      )

      for (s in seq_len(ncol(Y))) {
        y_true = Y[, s]
        y_pred = P_oof[, s]
        valid = !is.na(y_pred)

        if (sum(y_true[valid] == 1) >= 2 && sum(y_true[valid] == 0) >= 2) {
          species_cv$test_auc[s] = tryCatch(
            as.numeric(auc(roc(y_true[valid], y_pred[valid], quiet = TRUE))),
            error = function(e) NA
          )
        }

        species_cv$train_auc_mean[s] = mean(train_auc_per_fold[, s], na.rm = TRUE)
        species_cv$train_test_gap[s] = species_cv$train_auc_mean[s] - species_cv$test_auc[s]
      }

      species_cv = species_cv %>% arrange(test_auc)
      species_cv_file = paste0("exp4_species_cv_", param_tag, ".csv")
      write_csv(species_cv, file.path(drive_base, "results", species_cv_file))
      cat("Saved", species_cv_file, "\n")
      flush.console()

      cat("\nSpecies test AUC:\n")
      print(summary(species_cv$test_auc))
      cat("Train-test gap:\n")
      print(summary(species_cv$train_test_gap))
      flush.console()

      # --- Site metrics: log-loss ---
      eps = 1e-7
      site_cv = tibble(
        site = rownames(Y),
        richness = as.integer(richness),
        logloss = NA_real_
      )

      for (i in seq_len(nrow(Y))) {
        y_true = Y[i, ]
        y_pred = pmin(pmax(P_oof[i, ], eps), 1 - eps)
        if (all(!is.na(y_pred))) {
          site_cv$logloss[i] = -mean(y_true * log(y_pred) + (1 - y_true) * log(1 - y_pred))
        }
      }

      site_cv = site_cv %>% arrange(desc(logloss))
      site_cv_file = paste0("exp4_sites_cv_", param_tag, ".csv")
      write_csv(site_cv, file.path(drive_base, "results", site_cv_file))
      cat("Saved", site_cv_file, "\n")
      flush.console()

      cat("\nSite log-loss:\n")
      print(summary(site_cv$logloss))
      flush.console()
    }

## Final model with standard deviations of species betas

In [ ]:
# =============================================================================
  # Final model with se = TRUE (for species effect sizes + standard errors)
  # =============================================================================

  cat("Fitting final model with se = TRUE...\n")
  cat("Config:", rep_spatial, "\n")
  cat("Alpha:", rep_alpha, "| Lambda:", rep_lambda, "| Lambda_env:", rep_lambda_env, "\n")

  model_final = sjSDM(
    Y = Y,
    env = do.call(linear, list(
      data = Xenv,
      formula = env_formula,
      lambda = rep_lambda_env, alpha = rep_alpha
    )),
    spatial = do.call(linear, list(
      data = XY,
      formula = ~0 + Latitude + Longitude + I(Latitude * Longitude),
      lambda = rep_lambda, alpha = rep_alpha
    )),
    biotic = bioticStruct(lambda = rep_lambda, alpha = rep_alpha,
                          df = ncol(Y), reg_on_Cov = FALSE),
    iter = iterations,
    device = device,
    learning_rate = learning_rate,
    sampling = sampling_fit,
    control = sjSDMControl(
      RMSprop(weight_decay = 0.0),
      scheduler = 5L,
      early_stopping_training = 25L,
      lr_reduce_factor = 0.9
    ),
    se = TRUE
  )

  cat("Model fitted with se = TRUE\n")

  # Extract coefficients with SE
  coefs = coef(model_final)
  cat("Coefficient names:", paste(names(coefs), collapse = ", "), "\n")

  # Save
  param_tag = paste0("a", rep_alpha, "_l", rep_lambda, "_le", rep_lambda_env)
  saveRDS(model_final, file.path(drive_base, "results",
          paste0("final_model_se_", param_tag, ".rds")))
  cat("Saved final_model_se_", param_tag, ".rds\n")
  flush.console()


## Session info

In [ ]:
cat("\n=== All experiments complete ===")
print(session_info)
saveRDS(session_info, file.path(drive_base, "results",
                                paste0("session_info_", Sys.Date(), ".rds")))